# 🔴 Quiz 1: 나만의 역할 에이전트 설계
**PINE 2기 7회차 — Quiz 1 독립 실행 노트북**  
사용 기술: **Microsoft Agent Framework (MAF) 1.0.0**

---
## 📋 목표
`학생(Student)`, `전문가(Expert)`, `비평가(Critic)` 3개 에이전트로  
**"RAG 에이전트란 무엇인가?"** 주제로 GroupChat을 실행하세요.

## ⏱ 제한시간: 10분

## 🎯 성공 기준
- [ ] 3개 에이전트가 각각 다른 역할/성격을 가진다
- [ ] GroupChat이 총 6라운드 이상 실행된다
- [ ] 각 에이전트 응답이 역할에 맞게 다르다

## 💡 MAF GroupChatBuilder 핵심 패턴
```python
from agent_framework.orchestrations import GroupChatBuilder

def selector(state) -> str | None:
    if state["round_index"] >= MAX:  # None 반환 = 종료
        return None
    names = list(state["participants"].keys())
    return names[state["round_index"] % len(names)]

workflow = (
    GroupChatBuilder()
    .set_select_speakers_func(selector)
    .participants([agent1, agent2, agent3])
    .build()
)
result_messages = await workflow.run(task)  # Jupyter에서 await 직접 사용!
```

In [ ]:
# ⚙️ 환경 설정 (반드시 먼저 실행!)
import os
from dotenv import load_dotenv
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from agent_framework.orchestrations import GroupChatBuilder

load_dotenv()

chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_MODEL"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
)

# GroupChat 실행 헬퍼
async def run_groupchat(workflow, task: str):
    print(f"📢 주제: {task}")
    print("=" * 60)
    all_messages = []
    async for event in workflow.run(task, stream=True):
        if event.type == "output":
            data = event.data
            if hasattr(data, "text") and hasattr(data, "author_name"):
                print(f"\n🤖 [{data.author_name}]:")
                print(data.text)
                print("-" * 40)
                all_messages.append(data)
            elif isinstance(data, list):
                all_messages = data
    print(f"\n✅ 총 {len(all_messages)}개 메시지 완료")
    return all_messages

print("✅ 환경 설정 완료!")

In [ ]:
# ============================================================
# 🔴 Quiz 1: 나만의 역할 에이전트 설계
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 학생 에이전트: AI를 배우는 고등학생
student = Agent(
    client=chat_client,
    name="학생",
    instructions="""???
    # 포함할 내용: 역할, 말투(질문 많음), 답변 길이(2문장)
    """
)

# 2️⃣ 전문가 에이전트: AI 분야 경력자
expert = Agent(
    client=chat_client,
    name="전문가",
    instructions="""???
    # 포함할 내용: 역할, 정확한 설명, 비유 사용, 답변 길이(4~5문장)
    """
)

# 3️⃣ 비평가 에이전트: 논리적 허점 찾기
critic = Agent(
    client=chat_client,
    name="비평가",
    instructions="""???
    # 포함할 내용: 역할, 지적 + 개선 방향, 답변 길이(3문장)
    """
)

# 4️⃣ 발언자 선택 함수
TOTAL_ROUNDS = 6  # 에이전트당 2번 발언

def my_selector(state) -> str | None:
    if state["round_index"] >= ???:
        return None
    names = list(state["participants"].keys())
    return names[??? % len(names)]

# 5️⃣ GroupChat 구성
my_groupchat = (
    GroupChatBuilder()
    .set_select_speakers_func(???)
    .participants([???, ???, ???])
    .build()
)

# 6️⃣ 실행!
messages = await run_groupchat(
    my_groupchat,
    "RAG 에이전트란 무엇인가요? 오늘 배울 내용을 함께 정리해봐요!"
)

In [ ]:
# 🌟 자기 평가
print("=== 체크리스트 ===")
print(f"총 메시지 수: {len(messages)}개 (목표: 6개 이상)")
print()
print("각 에이전트가 역할에 맞게 답변했나요? system_message를 바꿔서 다시 실험해보세요!")

In [ ]:
# ✅ 정답 예시 (강사 참고용)

# student = Agent(client=chat_client, name="학생",
#     instructions="""당신은 AI에 관심 많은 고등학교 2학년 학생입니다.
# 전문 용어가 나오면 솔직하게 질문합니다. 이해한 내용을 비유로 표현하기도 합니다. 2문장으로."""
# )
# expert = Agent(client=chat_client, name="전문가",
#     instructions="""당신은 AI 분야 10년 경력 엔지니어입니다.
# 기술적으로 정확하되 고등학생이 이해할 수 있는 일상 비유를 사용합니다. 실제 사례 1개 포함. 4~5문장."""
# )
# critic = Agent(client=chat_client, name="비평가",
#     instructions="""당신은 비판적 사고를 하는 논리학자입니다.
# 앞선 설명의 허점이나 빠진 내용을 지적하고 개선 방향을 제시합니다. 3문장."""
# )
# def my_selector(state):
#     if state["round_index"] >= 6: return None
#     return list(state["participants"].keys())[state["round_index"] % 3]
# my_groupchat = GroupChatBuilder().set_select_speakers_func(my_selector).participants([student, expert, critic]).build()
# messages = await run_groupchat(my_groupchat, "RAG 에이전트란 무엇인가요?")